# Copernicus Marine Delivery – Python demo

In [1]:
import os
from pathlib import Path

for line in Path("../.env").read_text().splitlines():
    if line.startswith("export ") and "=" in line:
        key, value = line[len("export "):].split("=", 1)
        os.environ.setdefault(key, value.strip('"'))

# yaml file paths are relative to the CWD
os.chdir("delivery_multiple_operations")
os.environ.pop("PYTHON_LOG_DEBUG", None)

'true'

In [2]:
PRODUCT_ID = "GLOBAL_ANALYSISFORECAST_PHY_001_024"
DATASET_ID = "cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i_202406"

## 1. Build the delivery

A delivery is a list of operations that run in order. Delete 4 files, then upload 16 (4 of them re-use a deleted name).

In [7]:
from copernicusmarine_delivery import Delete, Delivery, Upload, delivery_status, list_deliveries

delivery = Delivery()

# 1st operation: delete files already published in the dataset (S3 key suffixes)
delivery.add(
    Delete(
        files=[
            "2026/05/glo12_rg_6h-i_20260528-18h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260529-00h_3D-uovo_nwct_R20260603.nc",
            "2026/06/glo12_rg_6h-i_20260602-06h_3D-uovo_nwct_R20260603.nc",
            "2026/06/glo12_rg_6h-i_20260602-12h_3D-uovo_nwct_R20260603.nc",
        ]
    )
)

# 2nd operation: upload local files (all .nc files below 2026/)
delivery.add(Upload(files=[str(f) for f in sorted(Path("2026").rglob("*.nc"))]))

for op in delivery.operations:
    print(type(op).__name__, len(op.files), "files")

Delete 4 files
Upload 4 files


## 2. Dry run

Validates product, dataset and files. Nothing is uploaded.

In [8]:
response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID, dry_run=True)
print("Fatal error:", response.fatal_error)

2026-10-29 14:10:28 - copernicusmarine-delivery - INFO - [DRY RUN]: Starting to upload 4 files ...
2026-10-29 14:10:28 - copernicusmarine-delivery - INFO - [DRY RUN]: Finished uploading 4 files for a total size of 29.00 MB.


Fatal error: None


## 3. Submit

In [9]:
from copernicusmarine_delivery import ResponseUpload, ResponseDelete


response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)

if response.fatal_error:
    print(f"Delivery failed: {response.fatal_error}")
else:
    print(f"Delivery ID: {response.delivery_id}")
    for r in response.operations_responses:
        if isinstance(r, ResponseUpload):
            print(f"  files uploaded: {len(r.files_uploaded)}")
            print(f"  files failed: {len(r.files_failed)}")
        if isinstance(r, ResponseDelete):
            print(f"  files to delete: {len(r.files_to_delete)}")

2026-10-29 14:10:32 - copernicusmarine-delivery - INFO - Starting to upload 4 files ...
2026-10-29 14:10:33 - copernicusmarine-delivery - INFO - Uploaded [1/4] file glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc
2026-10-29 14:10:33 - copernicusmarine-delivery - INFO - Uploaded [2/4] file glo12_rg_6h-i_20260528-00h_3D-uovo_nwct_R20260603.nc
2026-10-29 14:10:34 - copernicusmarine-delivery - INFO - Uploaded [3/4] file glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc
2026-10-29 14:10:37 - copernicusmarine-delivery - INFO - Uploaded [4/4] file glo12_rg_6h-i_20260527-18h_3D-uovo_nwct_R20260603.nc
2026-10-29 14:10:37 - copernicusmarine-delivery - INFO - Finished uploading 4 files for a total size of 29.00 MB.


Delivery ID: 20261029T141031-GLOBAL_ANALYSISFORECAST_PHY_001_024-2291
  files to delete: 4
  files uploaded: 4
  files failed: 0


## 4. Status

Reuses the `delivery_id` from the submit above.

In [15]:
import time

def wait_for_status(delivery_id: str):
    """Poll until the delivery is completed or failed, then show errors."""
    for _ in range(10):
        status = delivery_status(delivery_id)
        print(status.status.value)
        if status.status.value in ("completed", "failed"):
            break
        time.sleep(10)
    print(f"Delivery error: {status.error_message}" if status.error_message else "No errors in delivery!")
    for op in status.operations:
        for key_suffix, error in op.get_errored_file_names():
            print(f"  {op.operation.value} {key_suffix}: {error}")
    return status

DELIVERY_ID = response.delivery_id
status = wait_for_status(DELIVERY_ID)

completed
No errors in delivery!


## 5. List deliveries

In [16]:
for d in list_deliveries(timeframe="d"):
    print(d.delivery_id, d.status.value)

20261029T141031-GLOBAL_ANALYSISFORECAST_PHY_001_024-2291 completed
20261029T140928-GLOBAL_ANALYSISFORECAST_PHY_001_024-5810 completed
20261029T140904-GLOBAL_ANALYSISFORECAST_PHY_001_024-2870 completed
20261029T115340-GLOBAL_ANALYSISFORECAST_PHY_001_024-3942 completed
20261029T115039-GLOBAL_ANALYSISFORECAST_PHY_001_024-9293 completed
20261029T113913-GLOBAL_ANALYSISFORECAST_PHY_001_024-1859 completed
20261029T113201-GLOBAL_ANALYSISFORECAST_PHY_001_024-2104 completed
20261029T112147-GLOBAL_ANALYSISFORECAST_PHY_001_024-3252 completed
20261029T101623-GLOBAL_ANALYSISFORECAST_PHY_001_024-8840 completed
20261029T101230-GLOBAL_ANALYSISFORECAST_PHY_001_024-7777 completed
20261029T100838-GLOBAL_ANALYSISFORECAST_PHY_001_024-8424 completed
20261029T095529-GLOBAL_ANALYSISFORECAST_PHY_001_024-3900 completed
20261029T095212-GLOBAL_ANALYSISFORECAST_PHY_001_024-4056 completed
20261028T173220-GLOBAL_ANALYSISFORECAST_PHY_001_024-4602 completed
20261028T171301-GLOBAL_ANALYSISFORECAST_PHY_001_024-4706 compl

## Error cases

### A. Toolbox detects the error

Two of the files do not exist locally. The toolbox validates all files **before** uploading anything, so nothing is uploaded and no delivery is created.

In [ ]:
from copernicusmarine_delivery import InvalidFilesError

delivery_a = Delivery()
delivery_a.add(
    Upload(
        files=[
            "2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-12h_3D-uovo_nwct_R20260603.nc",
            "2026/05/glo12_rg_6h-i_20260527-18h_3D-uovo_nwct_R20260603.nc",
            "2026/05/this_file_does_not_exist_locally_1.nc",
            "2026/06/this_file_does_not_exist_locally_2.nc",
        ]
    )
)

try:
    delivery_a.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
except InvalidFilesError as error:
    print("Toolbox error:", error)

### B. OPDV detects the error

The toolbox cannot know what is in the dataset, so it accepts a delete of a file that does not exist. OPDV rejects it afterwards.

In [ ]:
delivery_b = Delivery()
delivery_b.add(Delete(files=["2026/05/glo12_rg_6h-i_20200101-00h_3D-uovo_nwct_DOES_NOT_EXIST.nc"]))
delivery_b.add(Upload(files=["2026/05/glo12_rg_6h-i_20260527-06h_3D-uovo_nwct_R20260603.nc"]))

response_b = delivery_b.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
print(f"Delivery ID: {response_b.delivery_id}")
status_b = wait_for_status(response_b.delivery_id)

### C. Delivery OK, error in publication

Toolbox and OPDV accept the delivery. The API is hardcoded to fail the publication of the file named `*_PUBERROR.nc`.

In [5]:
import os

from copernicusmarine_delivery import Delivery, Upload

os.chdir("../delivery_publication_error")

delivery_c = Delivery()
delivery_c.add(
    Upload(
        files=[
            "2026/06/glo12_rg_6h-i_20260603-00h_3D-uovo_nwct_R20260604.nc",
            "2026/06/glo12_rg_6h-i_20260603-06h_3D-uovo_nwct_R20260604.nc",
            "2026/06/glo12_rg_6h-i_20260603-12h_3D-uovo_nwct_PUBERROR.nc",
        ]
    )
)

response_c = delivery_c.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)
print(f"Delivery ID: {response_c.delivery_id}")
status_c = wait_for_status(response_c.delivery_id)

NameError: name 'PRODUCT_ID' is not defined